## DATA

### Visualize Sample Data

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, UnidentifiedImageError


DATA_FOLDER = "./app/modules/image_segmentation/modeling/data"

# Load image and corresponding annotation
image_filename = "Presc_0.jpg"
annotation_filename = "Presc_0.txt"

image_path = os.path.join(DATA_FOLDER, "train", "images", image_filename)
annotation_path = os.path.join(DATA_FOLDER, "train", "labels", annotation_filename)

try:
    # Load image
    img = Image.open(image_path).convert("RGB")
    img_w, img_h = img.size

    # Set figure DPI and size to match image for sharpness
    fig_w = img_w / 100  # inches
    fig_h = img_h / 100  # inches
    fig, ax = plt.subplots(1, figsize=(fig_w, fig_h), dpi=100)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title("Sample Validation Image with Annotations", fontsize=16)

    # Load and visualize annotation (YOLO format: class x_center y_center width height, normalized)
    if os.path.exists(annotation_path):
        with open(annotation_path, "r") as f:
            annotations = f.readlines()
        for ann in annotations:
            parts = ann.strip().split()
            if len(parts) != 5:
                continue
            cls, x_c, y_c, w, h = map(float, parts)
            # Convert normalized coordinates to pixel values
            x_c *= img_w
            y_c *= img_h
            w *= img_w
            h *= img_h
            # Calculate top-left corner
            x_min = x_c - w / 2
            y_min = y_c - h / 2
            # Draw rectangle with higher linewidth and alpha for clarity
            rect = patches.Rectangle(
                (x_min, y_min), w, h, 
                linewidth=3, edgecolor='lime', facecolor='none', alpha=0.9
            )
            ax.add_patch(rect)
    else:
        print(f"Annotation file not found: {annotation_path}")

    plt.tight_layout(pad=0)
    plt.show()

except UnidentifiedImageError:
    print(f"Cannot identify image file: {image_path}")
except FileNotFoundError:
    print(f"File not found: {image_path}")


### Clean Data

In [ ]:

# def clean_label_files(data_folder="./app/modules/image_segmentation/modeling/Data"):

#     label_dirs = [
#         os.path.join(data_folder, "train", "labels"),
#         os.path.join(data_folder, "valid", "labels")
#     ]

#     for label_dir in label_dirs:
#         if not os.path.exists(label_dir):
#             continue
#         for fname in os.listdir(label_dir):
#             fpath = os.path.join(label_dir, fname)
#             if not os.path.isfile(fpath):
#                 continue
#             new_lines = []
#             with open(fpath, "r") as f:
#                 for line in f:
#                     line_strip = line.strip()
#                     if not line_strip:
#                         continue
#                     parts = line_strip.split()
#                     if len(parts) == 0:
#                         continue
#                     # Only keep lines where class is 10, and change it to 0
#                     if parts[0] == "10":
#                         parts[0] = "0"
#                         new_lines.append(" ".join(parts) + "\n")
#             # Overwrite file with only the relabeled class 10 (now 0)
#             with open(fpath, "w") as f:
#                 f.writelines(new_lines)

# # clean_label_files()

# def rename_files_in_folder(folder_path, start_index):

#     images_path = os.path.join(folder_path, "images")
#     labels_path = os.path.join(folder_path, "labels")
    
#     if not os.path.exists(images_path):
#         return start_index
    
#     # Get all image files and sort for consistent ordering
#     image_files = [f for f in os.listdir(images_path) 
#                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
#     image_files.sort()
    
#     for file in image_files:
#         base_name = os.path.splitext(file)[0]
#         new_filename = f"Presc_{start_index}.jpg"
        
#         # Rename image file
#         old_image_path = os.path.join(images_path, file)
#         new_image_path = os.path.join(images_path, new_filename)
#         os.rename(old_image_path, new_image_path)
        
#         # Rename corresponding label file if it exists
#         old_label_path = os.path.join(labels_path, f"{base_name}.txt")
#         new_label_path = os.path.join(labels_path, f"Presc_{start_index}.txt")
        
#         if os.path.exists(old_label_path):
#             os.rename(old_label_path, new_label_path)
        
#         start_index += 1
    
#     return start_index

# # train_folder = os.path.join(DATA_FOLDER, "train")
# # valid_folder = os.path.join(DATA_FOLDER, "valid")

# # index = 0
# # index = rename_files_in_folder(train_folder, index)
# # index = rename_files_in_folder(valid_folder, index)
  

### Split Data

In [ ]:
# import os
# import shutil
# from sklearn.model_selection import train_test_split

# def split_data(data_folder, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    
#     assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1"
    
#     output_dirs = ['train', 'valid', 'test']
#     for split in output_dirs:
#         for subdir in ['images', 'labels']:
#             os.makedirs(os.path.join(data_folder, split, subdir), exist_ok=True)
    
#     # Get all image files
#     images_path = os.path.join(data_folder, "images")
#     if not os.path.exists(images_path):
#         print(f"Images folder not found at {images_path}")
#         return
    
#     image_files = [f for f in os.listdir(images_path) 
#                    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
    
#     if len(image_files) == 0:
#         print("No image files found")
#         return
    
#     print(f"Found {len(image_files)} image files")
    
#     train_files, temp_files = train_test_split(
#         image_files, 
#         train_size=train_ratio, 
#         random_state=42
#     )
    
#     val_files, test_files = train_test_split(
#         temp_files,
#         train_size=val_ratio/(val_ratio + test_ratio),
#         random_state=42
#     )
    
#     print(f"Train: {len(train_files)} files")
#     print(f"Validation: {len(val_files)} files") 
#     print(f"Test: {len(test_files)} files")
    
#     splits = {
#         'train': train_files,
#         'valid': val_files, 
#         'test': test_files
#     }
    
#     for split_name, files in splits.items():
#         for file in files:
#             base_name = os.path.splitext(file)[0]
            
#             src_image = os.path.join(images_path, file)
#             dst_image = os.path.join(data_folder, split_name, 'images', file)
#             shutil.copy2(src_image, dst_image)
            
#             src_label = os.path.join(data_folder, 'labels', f"{base_name}.txt")
#             dst_label = os.path.join(data_folder, split_name, 'labels', f"{base_name}.txt")
            
#             if os.path.exists(src_label):
#                 shutil.copy2(src_label, dst_label)
#             else:
#                 print(f"Warning: Label file not found for {file}")
    
#     print("Data splitting completed!")

# # DATA_FOLDER = "./app/modules/image_segmentation/modeling/data"
# # split_data(DATA_FOLDER, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1)


2025/09/12 09:54:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


Found 137 image files
Train: 95 files
Validation: 28 files
Test: 14 files
Data splitting completed!


## Train

In [ ]:
from ultralytics import YOLO
from ultralytics.cfg import get_cfg
from ultralytics import settings
settings.update({"mlflow": True})


cfg = get_cfg()
cfg.project = "./app/modules/image_segmentation/modeling/ultralytics_runs"  # change this to your desired root path

model = YOLO("/code/yolo11n.pt")

results = model.train(
    data="./app/modules/image_segmentation/modeling/data/data.yaml",
    epochs=1,
    device="mps",
    project=cfg.project  # pass the new root path here
)


New https://pypi.org/project/ultralytics/8.3.199 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.196 🚀 Python-3.12.11 torch-2.8.0+cpu CPU (aarch64)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./app/modules/image_segmentation/modeling/data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/code/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train23, nbs=64, nms

## Eval

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

test_image_path = "./app/modules/image_segmentation/modeling/data/valid/images/Presc1_jpg.rf.21a685ab301a0bb8e10ba6cc8577e51c.jpg"

results = model(test_image_path)
for result in results:
  result.show()